# Notebook 07 — TrendSignalStrategy Explained

This notebook walks through exactly what `TrendSignalStrategy(signal_df)` does — how it
reads the signal, how it sizes positions, and how it plugs into `BacktestEngine`.

**Key design choice:** each symbol always gets `1 / N_universe` weight when in signal.
If a symbol drops out, its slot becomes cash — other symbols are *not* resized upward.

In [3]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from hailmary.data.providers import YahooFinanceProvider
from hailmary.models import TrendSignal
from hailmary.backtest import BacktestEngine, TrendSignalStrategy
from hailmary.viz.theme import PALETTE, apply_theme

## 1. Build the Signal

Same setup as the other notebooks — fetch with warmup so MA-200 is valid from day one.

In [13]:
symbols = ["BTC-USD", "ETH-USD", "SOL-USD", "ADA-USD", "DOT-USD"]
start   = pd.Timestamp("2022-01-01")
end     = pd.Timestamp("2024-12-31")

signal = TrendSignal(ma_window=200)
yahoo  = YahooFinanceProvider()

fetch_start = start - pd.offsets.BDay(signal.warmup)
bars        = yahoo.get_bars(symbols, start=fetch_start, end=end)
signal_df   = signal.run(bars, trim_start=start)

print(f"Universe  : {symbols}")
print(f"N_universe: {len(symbols)}  →  max weight per symbol: {1/len(symbols):.1%}")
print(f"Bars      : {signal_df.index.get_level_values('timestamp').nunique()} trading days")

2026-04-26 18:14:15.616 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=cad45f214c5d


Universe  : ['BTC-USD', 'ETH-USD', 'SOL-USD', 'ADA-USD', 'DOT-USD']
N_universe: 5  →  max weight per symbol: 20.0%
Bars      : 1096 trading days


## 2. The Signal Matrix

`TrendSignalStrategy.__init__` immediately unstacks `signal_open` into a wide
**(date × symbol)** matrix of 0s and 1s.  This is the only data the strategy ever reads.

In [14]:
signal_matrix = signal_df["signal_open"].unstack(level="symbol")

print(f"Shape: {signal_matrix.shape}  (dates × symbols)")
print()
signal_matrix.head(10)

Shape: (1096, 5)  (dates × symbols)



symbol,ADA-USD,BTC-USD,DOT-USD,ETH-USD,SOL-USD
timestamp,,,,,
2022-01-01,0,0,0,1,1
2022-01-02,0,0,0,1,1
2022-01-03,0,0,1,1,1
2022-01-04,0,0,1,1,1
2022-01-05,0,0,0,1,1
2022-01-06,0,0,0,1,1
2022-01-07,0,0,0,1,1
2022-01-08,0,0,0,0,1
2022-01-09,0,0,0,0,1


In [15]:
# How many days each symbol spent in signal
signal_matrix.sum().rename("days_in_signal").to_frame().assign(
    pct=lambda df: (df["days_in_signal"] / len(signal_matrix) * 100).round(1)
)

,days_in_signal,pct
symbol,,
ADA-USD,310,28.3
BTC-USD,586,53.5
DOT-USD,290,26.5
ETH-USD,549,50.1
SOL-USD,512,46.7


## 3. Weight Generation — Step by Step

At each rebalance bar `generate_weights` does exactly three things:

1. Look up the row for `timestamp` in the signal matrix
2. Find which symbols have `signal_open == 1`
3. Assign `1 / N_universe` to each of them — the rest is cash

Let's replicate that manually for a few dates.

In [7]:
n_universe   = len(symbols)
fixed_weight = 1.0 / n_universe

sample_dates = signal_matrix.index[::120][:6]  # every ~6 months

rows = []
for ts in sample_dates:
    row    = signal_matrix.loc[ts]
    in_sig = row[row == 1].index.tolist()
    n_on   = len(in_sig)
    invested = n_on * fixed_weight
    rows.append({
        "date":          ts.date(),
        "in_signal":     ", ".join(in_sig) if in_sig else "(none)",
        "n_symbols_on":  n_on,
        "weight_each":   f"{fixed_weight:.1%}",
        "total_invested":f"{invested:.1%}",
        "cash":          f"{1 - invested:.1%}",
    })

pd.DataFrame(rows).set_index("date")

,in_signal,n_symbols_on,weight_each,total_invested,cash
date,,,,,
2022-01-01,"ETH-USD, SOL-USD",2,20.0%,40.0%,60.0%
2022-05-01,(none),0,20.0%,0.0%,100.0%
2022-08-29,(none),0,20.0%,0.0%,100.0%
2022-12-27,(none),0,20.0%,0.0%,100.0%
2023-04-26,"ADA-USD, BTC-USD, DOT-USD, ETH-USD, SOL-USD",5,20.0%,100.0%,0.0%
2023-08-24,SOL-USD,1,20.0%,20.0%,80.0%


## 4. Portfolio Allocation Over Time

Each symbol's weight is always `1/5 = 20%` when in signal, or `0%` when flat.
The cash line shows how much capital is sitting idle at each point.

In [8]:
weights_over_time = signal_matrix * fixed_weight
cash_over_time    = 1 - weights_over_time.sum(axis=1)

colours = [
    PALETTE["accent_blue"], PALETTE["accent_green"], PALETTE["accent_purple"],
    PALETTE["accent_orange"], PALETTE["accent_yellow"],
]

fig = go.Figure()

for i, sym in enumerate(symbols):
    fig.add_trace(go.Scatter(
        x=weights_over_time.index,
        y=weights_over_time[sym] * 100,
        name=sym,
        stackgroup="one",
        mode="lines",
        line={"width": 0.5},
        fillcolor=colours[i],
    ))

fig.add_trace(go.Scatter(
    x=cash_over_time.index,
    y=cash_over_time * 100,
    name="Cash",
    stackgroup="one",
    mode="lines",
    line={"width": 0.5},
    fillcolor="rgba(100,100,100,0.3)",
))

apply_theme(fig, "Portfolio Allocation Over Time", height=420)
fig.update_layout(yaxis_title="Allocation (%)", yaxis_range=[0, 100])
fig.show()

## 5. When Does a Trade Actually Fire?

The engine checks `generate_weights` on every trading day, but `set_weights` only
executes a trade when the target differs from the current position by more than a
rounding threshold.  So trades fire only on signal-flip days — let's count them.

In [ ]:
signal_changes = signal_matrix.diff().abs().sum(axis=1)
flip_days      = signal_changes[signal_changes > 0]
total_days     = len(signal_matrix)

print(f"Total trading days   : {total_days}")
print(f"Days with a flip     : {len(flip_days)}  ({len(flip_days)/total_days:.1%} of days)")
print(f"Quiet days (no trade): {total_days - len(flip_days)}")
print()

flip_days.value_counts().sort_index().rename("flip_days").rename_axis("symbols_changed")

## 6. Run the Engine

In [ ]:
engine = BacktestEngine(
    TrendSignalStrategy(signal_df),
    bars=bars,
    initial_capital=1_000_000,
)
result = engine.run()

print(f"Trades executed: {len(result.trade_log)}  (one per symbol per flip — verified against flip_days above)")
result.summary()

## 7. Weights Recorded by the Engine

`result.weights` records the target weights passed at each bar where `generate_weights`
was called.  Since the engine runs daily, every trading day appears — but only on flip
days do the values actually change.

In [11]:
print(f"Rebalance events recorded: {len(result.weights)}")
result.weights.head(10).fillna(0)

Rebalance events recorded: 1096


,ADA-USD,BTC-USD,DOT-USD,ETH-USD,SOL-USD
2022-01-01,0.0,0.0,0.0,0.2,0.2
2022-01-02,0.0,0.0,0.0,0.2,0.2
2022-01-03,0.0,0.0,0.2,0.2,0.2
2022-01-04,0.0,0.0,0.2,0.2,0.2
2022-01-05,0.0,0.0,0.0,0.2,0.2
2022-01-06,0.0,0.0,0.0,0.2,0.2
2022-01-07,0.0,0.0,0.0,0.2,0.2
2022-01-08,0.0,0.0,0.0,0.0,0.2
2022-01-09,0.0,0.0,0.0,0.0,0.2
2022-01-10,0.0,0.0,0.0,0.0,0.2


## 8. NAV vs Allocation

Overlay the NAV (normalised) with the total invested percentage to see how cash allocation
correlates with performance — periods of heavy cash are periods where the signal went flat.

In [12]:
nav_norm   = result.nav / result.nav.iloc[0] * 100
# reindex signal weights to match NAV dates
invested   = weights_over_time.sum(axis=1).reindex(nav_norm.index, method="ffill") * 100

fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.6, 0.4],
    subplot_titles=["NAV (normalised to 100)", "Total Invested (%)"],
    shared_xaxes=True,
    vertical_spacing=0.08,
)

fig.add_trace(go.Scatter(
    x=nav_norm.index, y=nav_norm,
    name="NAV", line={"color": PALETTE["accent_blue"], "width": 2},
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=invested.index, y=invested,
    name="Invested %", fill="tozeroy",
    line={"color": PALETTE["accent_green"], "width": 1.5},
    fillcolor="rgba(63,185,80,0.15)",
), row=2, col=1)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor=PALETTE["background"],
    plot_bgcolor=PALETTE["surface"],
    font={"color": PALETTE["text_primary"]},
    height=560,
    title={"text": "NAV vs Portfolio Allocation", "x": 0.02},
    margin={"l": 60, "r": 20, "t": 60, "b": 40},
)
for axis in fig.layout:
    if axis.startswith("xaxis") or axis.startswith("yaxis"):
        fig.layout[axis].update(gridcolor=PALETTE["border"])

fig.show()